# Worked solutions — XGBoost and model evaluation

These answers correspond to the exercises in `exercises.ipynb`. The bike rows are independent generated hourly scenarios, so results describe scenario interpolation/generalization under a random split, not forecasting across calendar time.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

SEED = 42
FEATURES = ["temperature_c", "humidity", "wind_kmh", "hour", "weekday", "season"]
TARGET = "rentals"

def find_data_file(filename="bike_demand.csv"):
    # In Jupyter, cwd is commonly the repository root or this notebook's folder.
    for start in (Path.cwd(), *Path.cwd().parents):
        candidate = start / "data" / filename
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find data/" + filename + " in the current directory or its parents. "
        "Open Jupyter from the repository or place the supplied CSV in data/."
    )

data_path = find_data_file()
df = pd.read_csv(data_path)
missing = sorted(set(FEATURES + [TARGET]) - set(df.columns))
if missing:
    raise ValueError(f"{data_path} is missing required columns: {missing}")
df = df[FEATURES + [TARGET]].dropna().copy()
print(f"Loaded {len(df):,} complete scenario rows from {data_path}")
df.head()


In [ ]:
# Treat hour and weekday as labels, not measurements with meaningful linear distance.
# One-hot columns are shared across all models so the comparison uses the same inputs.
categorical = ["hour", "weekday", "season"]
numeric = ["temperature_c", "humidity", "wind_kmh"]
encoder = ColumnTransformer(
    [("numeric", "passthrough", numeric),
     ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical)],
    verbose_feature_names_out=False,
)
X = df[FEATURES]
y = df[TARGET].astype(float)

# First hold out the final test set. Then split the remaining development rows
# into model-fit and early-stopping validation subsets.
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.1875, random_state=SEED
)  # 0.1875 * 0.80 = 0.15 of all rows
print(f"train={len(X_train)}, validation={len(X_val)}, test={len(X_test)}")


## Exercise 1 — Regularized leaf value

For \(H+\lambda>0\), the smooth quadratic has slope \(G+(H+\lambda)w\). The L1 term contributes subgradient \(\alpha\operatorname{sgn}(w)\) for nonzero \(w\), and interval \([-\alpha,\alpha]\) at zero. If \(G>\alpha\), the minimizer is negative and satisfies \(G+(H+\lambda)w-\alpha=0\); if \(G<-\alpha\), it is positive and satisfies \(G+(H+\lambda)w+\alpha=0\). If \(|G|\le\alpha\), zero satisfies the subgradient condition. Therefore

\[
w^*=-\frac{\operatorname{sgn}(G)(|G|-\alpha)_+}{H+\lambda}.
\]

Increasing \(\lambda\) shrinks the magnitude toward zero. The threshold \(\alpha\) can make the update exactly zero.

## Exercise 2 — Approximate split gain

The scores are \(S_L=64/12\), \(S_R=25/10\), and \(S_P=9/20\). Hence

\[
\tfrac12(S_L+S_R-S_P)-\gamma
=\tfrac12(5.3333+2.5-0.45)-0.4\approx3.2917.
\]

It is positive, so the split reduces this regularized second-order training approximation. That says nothing by itself about held-out generalization, stability, or usefulness under a different loss or data distribution.

## Exercise 3 — Fit and compare

The following code fits all models on the same encoded training rows. Only XGBoost sees validation rows during fitting. The final metrics are calculated on the untouched test rows.

In [ ]:
# Fit preprocessing only on training rows, then use that same representation everywhere.
X_train_enc = encoder.fit_transform(X_train)
X_val_enc = encoder.transform(X_val)
X_test_enc = encoder.transform(X_test)

dummy = DummyRegressor(strategy="mean").fit(X_train_enc, y_train)
forest = RandomForestRegressor(
    n_estimators=400, min_samples_leaf=2, max_features=0.9,
    random_state=SEED, n_jobs=-1,
).fit(X_train_enc, y_train)

# Validation data controls early stopping. The test data is untouched until scoring.
xgb = XGBRegressor(
    objective="reg:squarederror", eval_metric="rmse",
    n_estimators=2000, learning_rate=0.04, max_depth=4,
    min_child_weight=5, subsample=0.85, colsample_bytree=0.9,
    reg_lambda=3.0, reg_alpha=0.0, gamma=0.0,
    early_stopping_rounds=50,
    random_state=SEED, n_jobs=-1,
)
xgb.fit(X_train_enc, y_train, eval_set=[(X_val_enc, y_val)], verbose=False)
print("XGBoost best iteration:", xgb.best_iteration)


In [ ]:
def metric_row(name, model):
    pred = model.predict(X_test_enc)
    return {"model": name,
            "RMSE": mean_squared_error(y_test, pred) ** 0.5,
            "MAE": mean_absolute_error(y_test, pred),
            "R²": r2_score(y_test, pred)}

results = pd.DataFrame([
    metric_row("Training-mean baseline", dummy),
    metric_row("Random forest", forest),
    metric_row("XGBoost", xgb),
]).sort_values("RMSE").reset_index(drop=True)
results


Read this table as a final comparison of the three prespecified models. Do not use the test scores to choose a model or keep changing parameters; make any such choice from validation data, then evaluate the selected procedure on a new untouched test set.

## Exercise 4 — Residual diagnostics and scope

The plot below shows the XGBoost predictions on test examples. Residual summaries by hour can reveal structured errors that an overall score hides.

In [ ]:
pred = xgb.predict(X_test_enc)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_test, pred, alpha=0.35, s=14)
limits = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
axes[0].plot(limits, limits, "k--", linewidth=1)
axes[0].set(xlabel="actual rentals", ylabel="predicted rentals", title="XGBoost test predictions")
test_view = X_test.copy()
test_view["residual"] = y_test.to_numpy() - pred
by_hour = test_view.groupby("hour")["residual"].agg(["mean", "std"])
axes[1].errorbar(by_hour.index, by_hour["mean"], yerr=by_hour["std"], fmt="o", capsize=2)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set(xlabel="hour", ylabel="residual mean ± SD", title="Test residuals by hour")
plt.tight_layout()
by_hour

A suitable interpretation should describe the observed pattern in your run, for example whether residual means remain near zero across hours or show systematic underprediction at peak demand. Since rows are independent scenarios and there is no date ordering in the split, this result does not establish performance on later dates; a true forecasting study needs chronological splits and an explicitly time-indexed dataset.